In [29]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder  # ✅ Scaler not Scalar
import pickle

In [30]:
data=pd.read_csv("churn_modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15600001,Cook,407,France,Female,37,1,172407.88,1,1,0,5970.45,1
1,2,15600002,Mitchell,658,France,Male,42,10,178317.99,1,1,0,151758.56,1
2,3,15600003,Green,524,France,Male,45,1,101182.55,1,1,0,145944.00,0
3,4,15600004,King,390,Spain,Female,34,9,0.00,2,1,0,171058.79,0
4,5,15600005,King,492,France,Female,41,10,74074.68,2,0,0,32688.96,0


In [31]:
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,407,France,Female,37,1,172407.88,1,1,0,5970.45,1
1,658,France,Male,42,10,178317.99,1,1,0,151758.56,1
2,524,France,Male,45,1,101182.55,1,1,0,145944.00,0
3,390,Spain,Female,34,9,0.00,2,1,0,171058.79,0
4,492,France,Female,41,10,74074.68,2,0,0,32688.96,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,738,Spain,Female,45,9,48760.13,1,1,0,34755.64,0
9996,594,France,Female,34,5,68953.28,2,0,0,29690.36,1
9997,470,France,Male,47,2,142518.32,1,0,0,128312.23,0
9998,715,Germany,Male,42,4,0.00,1,0,0,81038.01,1


In [32]:
## encode categorical variable
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,407,France,0,37,1,172407.88,1,1,0,5970.45,1
1,658,France,1,42,10,178317.99,1,1,0,151758.56,1
2,524,France,1,45,1,101182.55,1,1,0,145944.00,0
3,390,Spain,0,34,9,0.00,2,1,0,171058.79,0
4,492,France,0,41,10,74074.68,2,0,0,32688.96,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,738,Spain,0,45,9,48760.13,1,1,0,34755.64,0
9996,594,France,0,34,5,68953.28,2,0,0,29690.36,1
9997,470,France,1,47,2,142518.32,1,0,0,128312.23,0
9998,715,Germany,1,42,4,0.00,1,0,0,81038.01,1


In [33]:
##one hot encode
from sklearn.preprocessing import OneHotEncoder

onehot_encoder_geo=OneHotEncoder()
geo_encoder=onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [34]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [35]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [36]:
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

In [38]:
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,1.0,0.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0
4,1.0,0.0,0.0
...,...,...,...
9995,0.0,0.0,1.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [39]:
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,407,0,37,1,172407.88,1,1,0,5970.45,1,1.0,0.0,0.0
1,658,1,42,10,178317.99,1,1,0,151758.56,1,1.0,0.0,0.0
2,524,1,45,1,101182.55,1,1,0,145944.00,0,1.0,0.0,0.0
3,390,0,34,9,0.00,2,1,0,171058.79,0,0.0,0.0,1.0
4,492,0,41,10,74074.68,2,0,0,32688.96,0,1.0,0.0,0.0


In [40]:
##save the encoders and scalar
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)

In [43]:
##dvide the dataset into dependent and indepenent features
x=data.drop('Exited',axis=1)
y=data['Exited']
 
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
scalar=StandardScaler()
x_train=scalar.fit_transform(x_train)
x_test=scalar.transform(x_test)

In [44]:
x_train

array([[ 1.68390139, -1.00375706,  1.13727361, ...,  0.99277609,
        -0.58273899, -0.56637961],
       [ 0.20659454, -1.00375706, -0.10070154, ...,  0.99277609,
        -0.58273899, -0.56637961],
       [ 1.2988214 , -1.00375706, -1.33867669, ..., -1.00727647,
         1.71603414, -0.56637961],
       ...,
       [ 0.06656545,  0.996257  ,  0.82777982, ...,  0.99277609,
        -0.58273899, -0.56637961],
       [-1.67679666,  0.996257  ,  0.41512144, ...,  0.99277609,
        -0.58273899, -0.56637961],
       [-1.44574867, -1.00375706, -1.23551209, ...,  0.99277609,
        -0.58273899, -0.56637961]], shape=(8000, 12))

In [45]:
with open('scalar.pkl','wb') as file:
    pickle.dump(scalar,file)

In [46]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,407,0,37,1,172407.88,1,1,0,5970.45,1,1.0,0.0,0.0
1,658,1,42,10,178317.99,1,1,0,151758.56,1,1.0,0.0,0.0
2,524,1,45,1,101182.55,1,1,0,145944.00,0,1.0,0.0,0.0
3,390,0,34,9,0.00,2,1,0,171058.79,0,0.0,0.0,1.0
4,492,0,41,10,74074.68,2,0,0,32688.96,0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,738,0,45,9,48760.13,1,1,0,34755.64,0,0.0,0.0,1.0
9996,594,0,34,5,68953.28,2,0,0,29690.36,1,1.0,0.0,0.0
9997,470,1,47,2,142518.32,1,0,0,128312.23,0,1.0,0.0,0.0
9998,715,1,42,4,0.00,1,0,0,81038.01,1,0.0,1.0,0.0


ANN Implementation


In [15]:
# Cell 1 — Imports
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime
from sklearn.model_selection import train_test_split

# Cell 2 — Load data
df = pd.read_csv('Churn_Modelling.csv')

# Cell 3 — Prepare data
# Drop irrelevant columns
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# Encode categorical columns
df = pd.get_dummies(df, columns=['Geography', 'Gender'], drop_first=True)

# Split features and target
X = df.drop('Exited', axis=1)   # 'Exited' is the target column (1=churned, 0=stayed)
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Cell 4 — Build model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  # Input layer
    Dense(32, activation='relu'),                                    # Hidden layer
    Dense(1, activation='sigmoid')                                   # Output layer
])

print(model.summary())

c:\Users\upput\Desktop\ANN\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

None


In [18]:
model.compile(optimizer='adam',          # ✅ Fixed empty optimizer
              loss='binary_crossentropy',
              metrics=['accuracy'])       # ✅ Fixed typo: metric → metrics

In [21]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

# Set up TensorFlow TensorBoard logging
log_dir = "log/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [24]:
early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [36]:
# Cell 1 — Imports
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from sklearn.model_selection import train_test_split
import datetime

# Cell 2 — Load data
df = pd.read_csv('Churn_Modelling.csv')

# Cell 3 — Prepare data
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
df = pd.get_dummies(df, columns=['Geography', 'Gender'], drop_first=True)

X = df.drop('Exited', axis=1)
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Cell 4 — Build model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.summary()

# Cell 5 — Compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Cell 6 — Callbacks
log_dir = "log/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Cell 7 — Train model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[tensorboard_callback, early_stopping_callback]
)

c:\Users\upput\Desktop\ANN\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6349 - loss: 352.1014 - val_accuracy: 0.4900 - val_loss: 59.9673
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6423 - loss: 105.1218 - val_accuracy: 0.7280 - val_loss: 51.2582
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6417 - loss: 129.6663 - val_accuracy: 0.7925 - val_loss: 108.2070
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6319 - loss: 111.0889 - val_accuracy: 0.7945 - val_loss: 138.3946
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6308 - loss: 115.3644 - val_accuracy: 0.7740 - val_loss: 56.0873
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6472 - loss: 77.6247 - val_accuracy: 0.6235 - val_loss: 41.9694
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6407 - loss: 89.2497 - val_accuracy: 0.6825 - val_loss: 34.8788
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6391 - lo

In [37]:
model.save('model.h5')


In [38]:
## Load Tensorboard Extension
%load_ext tensorboard


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [39]:
%tensorboard --logdir log/fit


ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\upput\Desktop\ANN\venv\Scripts\tensorboard.exe\__main__.py", line 2, in <module>
  File "C:\Users\upput\Desktop\ANN\venv\Lib\site-packages\tensorboard\main.py", line 27, in <module>
    from tensorboard import default
  File "C:\Users\upput\Desktop\ANN\venv\Lib\site-packages\tensorboard\default.py", line 30, in <module>
    import pkg_resources
ModuleNotFoundError: No module named 'pkg_resources'

In [ ]:
# Cell 1 — Imports
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from sklearn.model_selection import train_test_split
import datetime

# Cell 2 — Load data
df = pd.read_csv('Churn_Modelling.csv')

# Cell 3 — Prepare data
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
df = pd.get_dummies(df, columns=['Geography', 'Gender'], drop_first=True)

X = df.drop('Exited', axis=1)
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Cell 4 — Build model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.summary()

# Cell 5 — Compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Cell 6 — Callbacks
log_dir = "log/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Cell 7 — Train model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[tensorboard_callback, early_stopping_callback]
)

# Cell 8 — Save model
model.save('model.h5')

# Cell 9 — Launch TensorBoard
%load_ext tensorboard
%tensorboard --logdir log/fit

c:\Users\upput\Desktop\ANN\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6079 - loss: 267.1425 - val_accuracy: 0.7820 - val_loss: 66.4546
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6395 - loss: 120.1613 - val_accuracy: 0.5620 - val_loss: 87.0421
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6332 - loss: 131.3382 - val_accuracy: 0.6220 - val_loss: 55.5310
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6308 - loss: 124.8479 - val_accuracy: 0.7865 - val_loss: 80.8135
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6309 - loss: 130.0341 - val_accuracy: 0.7930 - val_loss: 52.0611
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6434 - loss: 140.5294 - val_accuracy: 0.7940 - val_loss: 88.1808
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6300 - loss: 96.7625 - val_accuracy: 0.7945 - val_loss: 107.6882
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6371 - lo

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\upput\Desktop\ANN\venv\Scripts\tensorboard.exe\__main__.py", line 2, in <module>
  File "C:\Users\upput\Desktop\ANN\venv\Lib\site-packages\tensorboard\main.py", line 27, in <module>
    from tensorboard import default
  File "C:\Users\upput\Desktop\ANN\venv\Lib\site-packages\tensorboard\default.py", line 30, in <module>
    import pkg_resources
ModuleNotFoundError: No module named 'pkg_resources'

In [4]:
# Cell 1 — Imports
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from sklearn.model_selection import train_test_split
import datetime

# Cell 2 — Load data
df = pd.read_csv('Churn_Modelling.csv')

# Cell 3 — Prepare data
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
df = pd.get_dummies(df, columns=['Geography', 'Gender'], drop_first=True)

X = df.drop('Exited', axis=1)
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Cell 4 — Build model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.summary()

# Cell 5 — Compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Cell 6 — Callbacks
log_dir = "log/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Cell 7 — Train model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[tensorboard_callback, early_stopping_callback]
)

# Cell 8 — Save model
model.save('model.h5')

# Cell 9 — Launch TensorBoard
%load_ext tensorboard
%tensorboard --logdir log/fit

c:\Users\upput\Desktop\ANN\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6340 - loss: 299.9885 - val_accuracy: 0.5610 - val_loss: 113.5174
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6374 - loss: 118.2060 - val_accuracy: 0.7945 - val_loss: 296.8008
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6432 - loss: 98.5660 - val_accuracy: 0.7945 - val_loss: 119.3477
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6400 - loss: 93.3051 - val_accuracy: 0.5585 - val_loss: 110.3595
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6351 - loss: 112.9298 - val_accuracy: 0.6195 - val_loss: 102.7001
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6439 - loss: 106.4682 - val_accuracy: 0.7500 - val_loss: 58.4957
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6423 - loss: 82.7300 - val_accuracy: 0.4545 - val_loss: 112.5243
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6355 -

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6007 (pid 31772), started 0:03:25 ago. (Use '!kill 31772' to kill it.)

In [ ]:
### load the pickle file


c:\Users\upput\Desktop\ANN\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6286 - loss: 182.5710 - val_accuracy: 0.4865 - val_loss: 130.0566
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6289 - loss: 114.2653 - val_accuracy: 0.6250 - val_loss: 52.0849
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6304 - loss: 96.7734 - val_accuracy: 0.4190 - val_loss: 109.7250
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6428 - loss: 74.9375 - val_accuracy: 0.6125 - val_loss: 29.3375
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6361 - loss: 84.5373 - val_accuracy: 0.7945 - val_loss: 56.8030
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6403 - loss: 71.1658 - val_accuracy: 0.6510 - val_loss: 53.0309
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6449 - loss: 68.2661 - val_accuracy: 0.7915 - val_loss: 52.6713
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6374 - loss:

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
